# 🚀 CLSS — Train & Test trên Subset 25%

**Pipeline:**
1. Clone repo + cài dependencies
2. Patch config YAML (đường dẫn Kaggle + `in_memory: false` để tránh OOM)
3. Train 10 epochs
4. Evaluate trên test set

**Fix OOM**: Dùng `in_memory: false` để dataset đọc từ disk thay vì load toàn bộ vào RAM.

## Cell 1 — Kiểm tra GPU

In [ ]:
!nvidia-smi
print()
!nvcc --version

## Cell 2 — Clone repo & cài dependencies

In [ ]:
import os
if not os.path.exists('/kaggle/working/CLSS'):
    !git clone https://github.com/thanhxuan217/CLSS.git
else:
    print('CLSS already cloned')
%cd /kaggle/working/CLSS
!pip install -r requirements.txt --upgrade -q

## Cell 3 — Cấu hình đường dẫn & kiểm tra

In [ ]:
import os

# ============================================================================
# CẤU HÌNH — CHỈNH SỬA NẾU CẦN
# ============================================================================
VIEW1_PATH = "/kaggle/input/datasets/xunhunh/subsettxt/data/sino_nom_punct_subset"
VIEW2_PATH = "/kaggle/input/datasets/xunhunh/subsettxt/data/sino_nom_punct_doc_subset"
SIKUBERT_PATH = "/kaggle/input/models/xunhunh/sikubertlocal/pytorch/default/1/pretrained/sikubert"
OUT_DIR = "/kaggle/working/output"

# ============================================================================
# KIỂM TRA
# ============================================================================
print("=" * 60)
all_ok = True
for name, path in [("View 1 subset", VIEW1_PATH),
                    ("View 2 subset", VIEW2_PATH),
                    ("SikuBERT", SIKUBERT_PATH)]:
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    status = "✅" if exists else "❌"
    print(f"  {status} {name}: {path}")
    if exists and os.path.isdir(path):
        for fn in sorted(os.listdir(path)):
            fp = os.path.join(path, fn)
            if os.path.isfile(fp):
                print(f"       {fn}: {os.path.getsize(fp)/1024:.0f} KB")
print("=" * 60)
print("✅ All OK!" if all_ok else "⛔ Fix paths above!")

## Cell 4 — Patch config YAML

Dùng **string replace** (KHÔNG dùng `yaml.safe_dump`) để giữ nguyên integer keys `{0: text, 1: punct}`.

Thêm `in_memory: false` để tránh OOM.

In [ ]:
import yaml

src = "config/sino_nom_doc_cl_subset.yaml"
dst = "config/sino_nom_doc_cl_kaggle.yaml"

with open(src, "r", encoding="utf-8") as f:
    content = f.read()

# String replace — giữ nguyên YAML format
replacements = {
    # Nếu config vẫn có đường dẫn cũ, thay thế
    "data/sino_nom_punct_subset": VIEW1_PATH,
    "data/sino_nom_punct_doc_subset": VIEW2_PATH,
    "/kaggle/input/datasets/xunhunh/subsettxt/data/sino_nom_punct_subset": VIEW1_PATH,
    "/kaggle/input/datasets/xunhunh/subsettxt/data/sino_nom_punct_doc_subset": VIEW2_PATH,
    "SIKU-BERT/sikubert": SIKUBERT_PATH,
    "resources/taggers/sino_nom_punct_tags.pkl": "/kaggle/working/CLSS/resources/taggers/sino_nom_punct_tags.pkl",
    "target_dir: resources/taggers/": f"target_dir: {OUT_DIR}/",
}
for old, new in replacements.items():
    content = content.replace(old, new)

# Đảm bảo in_memory: false có mặt
if "in_memory:" not in content:
    # Thêm in_memory: false sau mỗi data_folder
    content = content.replace(
        f"    data_folder: {VIEW1_PATH}",
        f"    data_folder: {VIEW1_PATH}\n    in_memory: false"
    )
    content = content.replace(
        f"    data_folder: {VIEW2_PATH}",
        f"    data_folder: {VIEW2_PATH}\n    in_memory: false"
    )

with open(dst, "w", encoding="utf-8") as f:
    f.write(content)

# Verify
with open(dst, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

v1 = cfg['punct']['ColumnCorpus-SinoNomPunct']
v2 = cfg['punct']['ColumnCorpus-SinoNomPunctDoc']
print("📝 Config verified:")
print(f"  View 1: {v1['data_folder']}")
print(f"  View 1 in_memory: {v1.get('in_memory', 'NOT SET ⚠️')}")
print(f"  View 2: {v2['data_folder']}")
print(f"  View 2 in_memory: {v2.get('in_memory', 'NOT SET ⚠️')}")
print(f"  column_format keys: {list(v1['column_format'].keys())} (types: {[type(k).__name__ for k in v1['column_format'].keys()]})")
print(f"  Transformer: {cfg['embeddings']['TransformerWordEmbeddings-0']['model']}")
print(f"  Epochs: {cfg['train']['max_epochs']}, Batch: {cfg['train']['mini_batch_size']}, LR: {cfg['train']['learning_rate']}")
print(f"  Output: {cfg['target_dir']}")

assert all(isinstance(k, int) for k in v1['column_format'].keys()), "column_format keys must be int!"
assert v1.get('in_memory') == False, "in_memory must be false!"
print("\n✅ Config OK!")

## Cell 5 — 🚀 TRAINING

In [ ]:
import subprocess, sys

print("🚀 Training...")
print(f"   Config: config/sino_nom_doc_cl_kaggle.yaml")
print(f"   Model:  {SIKUBERT_PATH}")
print(f"   Output: {OUT_DIR}")
print("=" * 60, flush=True)

retcode = subprocess.call([
    sys.executable, 'train.py',
    '--config', 'config/sino_nom_doc_cl_kaggle.yaml',
    '--transformer_model', SIKUBERT_PATH,
    '--out_dir', OUT_DIR
])

print(f'\n{"✅" if retcode == 0 else "❌"} Training finished (code: {retcode})')

## Cell 6 — 📊 EVALUATION

In [ ]:
import subprocess, sys

print("📊 Evaluating on test set...")
print("=" * 60, flush=True)

retcode = subprocess.call([
    sys.executable, 'train.py',
    '--config', 'config/sino_nom_doc_cl_kaggle.yaml',
    '--test',
    '--transformer_model', SIKUBERT_PATH,
    '--out_dir', OUT_DIR
])

print(f'\n{"✅" if retcode == 0 else "❌"} Evaluation finished (code: {retcode})')

## Cell 7 — Xem kết quả

In [ ]:
import os, glob

# List output files
print("📁 Output files:")
print("=" * 60)
if os.path.exists(OUT_DIR):
    for root, dirs, files in os.walk(OUT_DIR):
        level = root.replace(OUT_DIR, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        for file in sorted(files):
            fp = os.path.join(root, file)
            print(f'{indent}  {file} ({os.path.getsize(fp)/1024/1024:.1f} MB)')
else:
    print(f"  ❌ {OUT_DIR} not found")

# Training log
log_file = os.path.join(OUT_DIR, 'training.log')
if os.path.exists(log_file):
    print("\n📋 Training log (last 50 lines):")
    print("=" * 60)
    with open(log_file, 'r') as f:
        for line in f.readlines()[-50:]:
            print(line.rstrip())